# NMF でトピックを抽出する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/11_topics_keywords/nmf_topics.ipynb)

NLP クックブック 11-06 ／ トピックとキーワードの抽出

LDA より速く、短い文書でも安定しやすい方法でトピックを抽出したい。TF-IDF 行列を非負値行列因子分解する。

In [1]:
from collections import Counter
from sklearn.datasets import fetch_20newsgroups
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

names = fetch_20newsgroups(subset="train").target_names
cats = [c for c in names if not c.startswith("talk.politics")]   # 政治の話題のグループは除く
data = fetch_20newsgroups(subset="train", categories=cats, remove=("headers", "footers", "quotes"))

# 署名のように、5 件以上の投稿に同じ形で現れる行を除く
def lines(d):
    return {l.strip() for l in d.split("\n") if len(l.strip()) > 10}
count = Counter(l for d in data.data for l in lines(d))
repeated = {l for l, n in count.items() if n >= 5}
def clean(d):
    return "\n".join(l for l in d.split("\n") if l.strip() not in repeated)

docs = [c for c in map(clean, data.data) if len(c) > 200]
vec = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5, max_features=20000, token_pattern=r"(?u)\b[a-zA-Z]{3,}\b")
X = vec.fit_transform(docs)

nmf = NMF(n_components=12, init="nndsvd", random_state=0, max_iter=400)
W = nmf.fit_transform(X)      # 文書 × トピックの重み
H = nmf.components_           # トピック × 単語の重み
vocab = vec.get_feature_names_out()
for k, row in enumerate(H):
    print(f"{k:2} {' '.join(vocab[i] for i in row.argsort()[::-1][:9])}")

 0 don just people think like know good time really
 1 god jesus bible christ faith believe christian christians church
 2 windows dos file files program use running using run
 3 drive disk drives hard floppy boot power controller cable
 4 key chip clipper keys encryption government escrow algorithm public
 5 team game games year players hockey season play win
 6 card video monitor bus drivers cards vga ram memory
 7 mail edu thanks email com information send info address
 8 car bike new cars engine miles price dealer good
 9 window motif application server display widget manager problem using
10 space nasa launch shuttle orbit earth moon data lunar
11 scsi ide controller bus mac isa bit devices sec


In [2]:
# 各トピックを最もよく代表する文書
for k in [3, 5, 8]:
    i = W[:, k].argmax()
    print(k, "|", " ".join(docs[i].split())[:110])

3 | Hi, I just disassembled my old XT and get 2 disk drives - 30M hard drive and a 360K floppy drive. My questions
5 | Archive-name: hockey-faq rec.sport.hockey answers to Frequently Asked Questions and other news: Contents: 0. N
8 | If you don't already know it, you should call the bank/credit union/ finance company that holds the loan on yo


- TF-IDF 行列 X を、非負の 2 つの行列の積 W × H で近似する。非負なので「足し算だけ」で文書が表現され、解釈しやすい。
- 確率モデルではないので、LDA のような確率的な解釈はできないが、決定的 (`init="nndsvd"`) で再現性が高い。
- ツイートやタイトルのような短文には、LDA より NMF のほうが良い結果になることが多い。
- 署名や定型文のように同じ文字列を繰り返す投稿者がいると、その語だけのトピックができてしまう。`remove=("headers", "footers", "quotes")` でも署名は取り切れないので、多くの投稿に出てくる行を除いた。